# Live Soccer Win Probability Forecasting

**Stochastic modeling with Poisson processes, continuous-time Markov chains, MLE, Skellam distributions, time-varying intensities, Monte Carlo simulation, and xG.**

This notebook presents the research workflow behind the repository. The goal is to estimate match-outcome probabilities from a transparent stochastic baseline and then show how the framework extends to richer event-level information.

## 1. Problem setup

Let $N_T(t)$ and $N_O(t)$ denote goals scored by a team and its opponent by time $t$. Under the baseline model,

$$N_T(t) \sim \mathrm{Poisson}(\lambda_T t), \qquad N_O(t) \sim \mathrm{Poisson}(\lambda_O t),$$

with independent increments. The score differential

$$D(t)=N_T(t)-N_O(t)$$

is therefore a birth-death continuous-time Markov chain. This gives us two complementary ways to compute outcome probabilities:

1. numerically evolve the CTMC using the Kolmogorov forward equations;
2. use the exact Skellam distribution as a benchmark.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.soccer_winprob import (
    build_generator,
    estimate_lambda_kernel,
    forward_euler,
    outcome_from_distribution,
    outcome_curve,
    simulate_remaining_match,
    skellam_outcome,
)

## 2. Maximum-likelihood calibration

For a homogeneous Poisson process, if a team scores $G$ goals across $n$ matches of length $T$, the MLE is

$$\hat\lambda=\frac{G}{nT}.$$

The original case study filtered **372 Corinthians Série A matches** and estimated separate home/away rates. The aggregate rates are stored in the repository so the stochastic model is reproducible without redistributing the original third-party dataset.

In [ ]:
rates = pd.read_csv(ROOT / "data" / "baseline_rates.csv")
rates

In [ ]:
home = rates.loc[rates["context"] == "home"].iloc[0]
lambda_team = float(home["team_rate_per_min"])
lambda_opp = float(home["opponent_rate_per_min"])

print(f"Team intensity:     {lambda_team:.6f} goals/min ({90*lambda_team:.2f} per 90)")
print(f"Opponent intensity: {lambda_opp:.6f} goals/min ({90*lambda_opp:.2f} per 90)")

## 3. CTMC representation

For score-difference state $d$,

$$d\to d+1 \text{ at rate } \lambda_T, \qquad d\to d-1 \text{ at rate } \lambda_O.$$

Writing $p(t)$ for the state-probability vector and $Q$ for the generator,

$$p'(t)=Q^\top p(t).$$

We truncate the state space to a wide interval and integrate with forward Euler.

In [ ]:
Q, states = build_generator(lambda_team, lambda_opp, d_min=-10, d_max=10)

p0 = np.zeros(len(states))
p0[np.where(states == 0)[0][0]] = 1.0

times, trajectory = forward_euler(Q, p0, horizon=90.0, dt=0.05)
ctmc = outcome_from_distribution(trajectory[-1], states)
exact = skellam_outcome(lambda_team, lambda_opp, horizon=90.0)

comparison = pd.DataFrame({
    "CTMC / Euler": ctmc.as_array(),
    "Skellam benchmark": exact.as_array(),
    "absolute error": np.abs(ctmc.as_array() - exact.as_array()),
}, index=["Win", "Draw", "Loss"])
comparison

The baseline home model yields roughly **53.4% win / 26.3% draw / 20.3% loss**. The CTMC and Skellam calculations agree closely, which serves as a numerical correctness check.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(states, trajectory[-1])
plt.xlabel("Final score differential")
plt.ylabel("Probability")
plt.title("Final Score-Difference Distribution")
plt.show()

## 4. Live probability curves

A live forecast should condition on the **current score state** and the **time remaining**. The figure below answers a clean counterfactual: *if the match is still 0–0 at minute $t$, what are the model-implied final win/draw/loss probabilities?*

As the remaining horizon shrinks, the draw probability rises toward 1 because there is progressively less time for either team to score.

In [ ]:
minute_grid, curve = outcome_curve(
    lambda_team, lambda_opp, horizon=90, current_diff=0
)

plt.figure(figsize=(9, 5))
plt.plot(minute_grid, curve[:, 0], label="Win")
plt.plot(minute_grid, curve[:, 1], label="Draw")
plt.plot(minute_grid, curve[:, 2], label="Loss")
plt.xlabel("Match minute (conditional on score remaining 0–0)")
plt.ylabel("Final outcome probability")
plt.title("Live W/D/L Forecast as Remaining Time Shrinks")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 5. Time-varying scoring intensity

A homogeneous Poisson model assumes constant scoring intensity. To relax this, historical goal times can be smoothed with a Gaussian kernel to estimate a non-homogeneous intensity $\lambda(t)$. This captures broad temporal structure such as late-game increases in scoring activity.

In [ ]:
# Demonstration with synthetic goal times; replace with historical event-level goal times.
rng = np.random.default_rng(11)
example_goal_times = np.clip(rng.normal(loc=55, scale=24, size=220), 0, 90)
t_grid, lambda_t = estimate_lambda_kernel(
    example_goal_times,
    total_matches=150,
    bandwidth=4.0,
)

plt.figure(figsize=(9, 4))
plt.plot(t_grid, lambda_t)
plt.xlabel("Match minute")
plt.ylabel("Estimated goal intensity")
plt.title("Gaussian-Kernel Estimate of Time-Varying Goal Intensity")
plt.grid(alpha=0.25)
plt.show()

## 6. Event-level xG and Monte Carlo extension

The project also explores StatsBomb Open Data, which contains timestamped shots, xG, goals, cards, substitutions, and other match events. A production-style live model can use these inputs to update scoring probabilities as the game evolves.

A generic feature vector at minute $t$ may include:

- elapsed time and score differential;
- cumulative and recent xG differential;
- red-card / player differential;
- recent shot volume and attacking pressure;
- pre-match attacking and defensive strength.

Once a model outputs minute-level scoring probabilities $p_T(t)$ and $p_O(t)$, Monte Carlo simulation turns those hazards into final W/D/L probabilities.

In [ ]:
# Example: 30 minutes remain and the team is currently leading by one goal.
remaining_minutes = 30
team_p = np.repeat(0.014, remaining_minutes)
opp_p = np.repeat(0.018, remaining_minutes)

mc = simulate_remaining_match(
    team_p,
    opp_p,
    n_sim=50_000,
    current_diff=1,
    seed=42,
)
mc

### Optional StatsBomb ingestion

The following pattern fetches open event data when an internet connection is available:

```python
from statsbombpy import sb

matches = sb.matches(competition_id=2, season_id=27)
events = sb.events(match_id=<MATCH_ID>)
shots = events.loc[events["type"] == "Shot", [
    "minute", "team", "player", "shot_outcome", "shot_statsbomb_xg"
]]
```

The original analysis used this event stream to inspect goals, substitutions, red cards, and shot-level xG before designing the event-aware extension.

## 7. Validation and limitations

**Validation.** The numerical generator satisfies the row-sum condition, probability mass is conserved, and the CTMC solver agrees with the exact Skellam benchmark under constant intensities.

**Limitations.** Independence and constant-rate assumptions are strong; soccer scoring depends on match state, team strength, tactical changes, and event history. The event-aware/xG section is therefore best interpreted as an extension path rather than a fully calibrated production forecasting system.

**Next steps.** Fit minute-level hazard models on a larger event dataset, calibrate probabilities out of sample, evaluate Brier/log loss, and compare against bookmaker-implied probabilities or strong statistical baselines.